# MWE Token Extraction Debug Notebook

This notebook is designed to debug issues with Multi-Word Expression (MWE) token extraction where the system is incorrectly taking all tokens from start to end of MWE instead of just the actual MWE tokens.

## Problem Description
- Instead of taking only tokens that belong to an MWE
- The system takes ALL tokens from the start position to the end position 
- This includes tokens in the middle that are NOT part of the MWE
- We need to identify where this error occurs in preprocessing or processing

## 1. Setup and Import Libraries

In [1]:
import pandas as pd
import logging
import sys
import os
from typing import List, Dict, Any

# Add project root to path to import local modules
sys.path.append(r'e:\Github\LexiSense-SR')

# Import local modules
try:
    from preprocessing import (
        get_mwe_tokens, 
        mark_mwe, 
        mark_tokens, 
        collect_mwe_senses
    )
    from webanno_spacy_converter.models.annotation_token import AnnotationToken
    from webanno_spacy_converter.models.sentence_with_mwes import (
        AnnotatedSentenceWithMWEs,
        MultiWordExpression,
    )
    # Import the correct function from data_loader
    from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser
    print("✅ All modules imported successfully!")
except ImportError as e:
    print(f"❌ Error importing modules: {e}")
    print("Make sure the project dependencies are installed and paths are correct.")

# Setup logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

# Display versions and setup info
print(f"Python version: {sys.version}")
print(f"Working directory: {os.getcwd()}")
print(f"Pandas version: {pd.__version__}")

✅ All modules imported successfully!
Python version: 3.12.1 (tags/v3.12.1:2305ca5, Dec  7 2023, 22:03:25) [MSC v.1937 64 bit (AMD64)]
Working directory: e:\Github\LexiSense-SR
Pandas version: 2.2.3


## 2. Load Sample Data

Let's load the example-mwe.tsv file and examine its structure to understand the WebAnno TSV format and identify MWE annotations.

In [2]:
# Load the example TSV file
tsv_file_path = r'e:\Github\LexiSense-SR\Data\example-mwe.tsv'

print(f"Loading file: {tsv_file_path}")
print(f"File exists: {os.path.exists(tsv_file_path)}")

# Read raw TSV content to examine structure
with open(tsv_file_path, 'r', encoding='utf-8') as f:
    raw_content = f.read()

print("\n" + "="*50)
print("RAW TSV CONTENT (first 2000 characters):")
print("="*50)
print(raw_content[:2000])

# Split into lines and examine header information
lines = raw_content.split('\n')
print(f"\nTotal lines in file: {len(lines)}")

# Show first 20 lines to understand structure
print("\n" + "="*50)
print("FIRST 20 LINES:")
print("="*50)
for i, line in enumerate(lines[:20]):
    print(f"{i+1:2d}: {line}")

Loading file: e:\Github\LexiSense-SR\Data\example-mwe.tsv
File exists: True

RAW TSV CONTENT (first 2000 characters):
#FORMAT=WebAnno TSV 3.3
#T_SP=de.tudarmstadt.ukp.dkpro.core.api.lexmorph.type.pos.POS|PosValue|coarseValue
#T_SP=de.tudarmstadt.ukp.dkpro.core.api.ner.type.NamedEntity|identifier|value
#T_SP=de.tudarmstadt.ukp.dkpro.core.api.segmentation.type.Lemma|value
#T_SP=webanno.custom.MWE|MWEid|MWElemma|MWEtype
#T_SP=webanno.custom.WSD|Comment|Explanation|KBid|NumberOfSenses|Origine|Possible


#Text=Bakterije se mogu širiti i putem krvi.
185-1	17460-17469	Bakterije	N	NOUN	_	_	bakterija	_	_	_	*	U ovom kontekstu, 'bakterije' se odnosi na mikroorganizme koji mogu da se šire putem krvi, što odgovara opisu jednoćelijskih organizama iz ponuđenog značenja.	http://llod.jerteh.rs/WSD/seENG3001348530n	1	ChatGPT	ENG30-01348530-n	
185-2	17470-17472	se	PAR	PART	_	_	se	1	*	IRV	*[607]	U ovom kontekstu, 'širiti se' znači da bakterije postaju rasprostranjenije i zauzimaju veći prostor u organizmu

## 3. Parse WebAnno TSV Format

Let's parse the TSV file using the data loader and examine the parsed sentence structures.

In [3]:
# Load sentences using the WebAnno parser directly
try:
    # Use the WebAnno parser to load the TSV file
    parser = WebAnnoLEXISParser(tsv_file_path)
    sentences = parser.parse()
    print(f"✅ Successfully loaded {len(sentences)} sentences")
    
    # Examine the first sentence in detail
    if sentences:
        sentence = sentences[0]
        print(f"\n" + "="*50)
        print("FIRST SENTENCE ANALYSIS:")
        print("="*50)
        print(f"Text: {sentence.text}")
        print(f"Number of tokens: {len(sentence.tokens)}")
        print(f"Number of MWEs: {len(sentence.mwes)}")
        
        # Display all tokens
        print(f"\n--- ALL TOKENS ---")
        for i, token in enumerate(sentence.tokens):
            print(f"Token {i}: '{token.text}' [{token.start}:{token.end}] layers: {token.layers}")
        
        # Display all MWEs
        print(f"\n--- ALL MWEs ---")
        for i, mwe in enumerate(sentence.mwes):
            print(f"MWE {i}:")
            print(f"  Lemma: {mwe.lemma}")
            print(f"  Type: {mwe.type}")
            print(f"  Token indices: {mwe.token_indices}")
            
            # Show which tokens these indices correspond to
            mwe_tokens = [sentence.tokens[idx] for idx in mwe.token_indices]
            print(f"  Actual tokens:")
            for idx, token_idx in enumerate(mwe.token_indices):
                token = sentence.tokens[token_idx]
                print(f"    {idx}: Token[{token_idx}] = '{token.text}' [{token.start}:{token.end}]")
                
except Exception as e:
    print(f"❌ Error loading sentences: {e}")
    import traceback
    traceback.print_exc()

✅ Successfully loaded 2 sentences

FIRST SENTENCE ANALYSIS:
Text: Bakterije se mogu širiti i putem krvi.
Number of tokens: 8
Number of MWEs: 1

--- ALL TOKENS ---
Token 0: 'Bakterije' [0:9] layers: {'PosValue': 'N', 'coarseValue': 'NOUN', 'value_4': 'bakterija', 'Comment': '*', 'Explanation': "U ovom kontekstu, 'bakterije' se odnosi na mikroorganizme koji mogu da se šire putem krvi, što odgovara opisu jednoćelijskih organizama iz ponuđenog značenja.", 'KBid': 'http://llod.jerteh.rs/WSD/seENG3001348530n', 'NumberOfSenses': '1', 'Origine': 'ChatGPT', 'Possible': 'ENG30-01348530-n'}
Token 1: 'se' [10:12] layers: {'PosValue': 'PAR', 'coarseValue': 'PART', 'value_4': 'se', 'MWEid': '1', 'MWElemma': '*', 'MWEtype': 'IRV', 'Comment': '*[607]', 'Explanation': "U ovom kontekstu, 'širiti se' znači da bakterije postaju rasprostranjenije i zauzimaju veći prostor u organizmu, što odgovara značenju 'uzimati maha, dobijati sve veće razmere, postajati se rasprostranjeniji'.[607]", 'KBid': 'http://llod

## 4. Debug MWE Token Extraction

Now let's test the `get_mwe_tokens()` function to see if it's correctly extracting only MWE tokens or including intermediate tokens.

In [4]:
# Test get_mwe_tokens function for each MWE
if sentences:
    for sent_idx, sentence in enumerate(sentences):
        print(f"\n{'='*60}")
        print(f"SENTENCE {sent_idx + 1}: {sentence.text}")
        print(f"{'='*60}")
        
        if not sentence.mwes:
            print("No MWEs found in this sentence")
            continue
            
        for mwe_idx, mwe in enumerate(sentence.mwes):
            print(f"\n--- MWE {mwe_idx + 1} ---")
            print(f"Lemma: '{mwe.lemma}'")
            print(f"Type: '{mwe.type}'")
            print(f"Token indices: {mwe.token_indices}")
            
            # Test get_mwe_tokens function
            try:
                extracted_tokens = get_mwe_tokens(sentence, mwe)
                print(f"\n🔍 EXTRACTED TOKENS (via get_mwe_tokens):")
                for i, token in enumerate(extracted_tokens):
                    print(f"  {i}: '{token.text}' [{token.start}:{token.end}]")
                
                # Compare with manual extraction using indices
                print(f"\n🔍 MANUAL TOKEN EXTRACTION (using indices):")
                manual_tokens = [sentence.tokens[idx] for idx in mwe.token_indices]
                for i, token in enumerate(manual_tokens):
                    print(f"  {i}: '{token.text}' [{token.start}:{token.end}]")
                
                # Check if they match
                tokens_match = len(extracted_tokens) == len(manual_tokens)
                if tokens_match:
                    for ext_token, man_token in zip(extracted_tokens, manual_tokens):
                        if ext_token.text != man_token.text or ext_token.start != man_token.start:
                            tokens_match = False
                            break
                
                print(f"\n✅ Token extraction matches manual: {tokens_match}")
                
                # Show the full span that would be covered if taking all tokens from start to end
                if mwe.token_indices:
                    min_idx = min(mwe.token_indices)
                    max_idx = max(mwe.token_indices)
                    print(f"\n🚨 POTENTIAL ISSUE - ALL TOKENS FROM {min_idx} to {max_idx}:")
                    for idx in range(min_idx, max_idx + 1):
                        token = sentence.tokens[idx]
                        is_mwe_token = idx in mwe.token_indices
                        marker = "✓" if is_mwe_token else "❌"
                        print(f"  {marker} Token[{idx}]: '{token.text}' [{token.start}:{token.end}]")
                
            except Exception as e:
                print(f"❌ Error extracting tokens: {e}")
                import traceback
                traceback.print_exc()


SENTENCE 1: Bakterije se mogu širiti i putem krvi.

--- MWE 1 ---
Lemma: 'širiti se'
Type: 'IRV'
Token indices: [1, 3]

🔍 EXTRACTED TOKENS (via get_mwe_tokens):
  0: 'se' [10:12]
  1: 'širiti' [18:24]

🔍 MANUAL TOKEN EXTRACTION (using indices):
  0: 'se' [10:12]
  1: 'širiti' [18:24]

✅ Token extraction matches manual: True

🚨 POTENTIAL ISSUE - ALL TOKENS FROM 1 to 3:
  ✓ Token[1]: 'se' [10:12]
  ❌ Token[2]: 'mogu' [13:17]
  ✓ Token[3]: 'širiti' [18:24]

SENTENCE 2: Simptomi se obično poboljšavaju u roku od dva dana, ali mogu trajati i do sedam dana.

--- MWE 1 ---
Lemma: 'poboljšavati se'
Type: 'IRV'
Token indices: [1, 3]

🔍 EXTRACTED TOKENS (via get_mwe_tokens):
  0: 'se' [9:11]
  1: 'poboljšavaju' [19:31]

🔍 MANUAL TOKEN EXTRACTION (using indices):
  0: 'se' [9:11]
  1: 'poboljšavaju' [19:31]

✅ Token extraction matches manual: True

🚨 POTENTIAL ISSUE - ALL TOKENS FROM 1 to 3:
  ✓ Token[1]: 'se' [9:11]
  ❌ Token[2]: 'obično' [12:18]
  ✓ Token[3]: 'poboljšavaju' [19:31]

--- MWE 2 -

## 5. Visualize Token Boundaries

Let's create visualizations to better understand token positions and MWE spans.

In [5]:
def visualize_sentence_tokens(sentence, show_positions=True):
    """Visualize token boundaries and positions in a sentence."""
    text = sentence.text
    print(f"Sentence: '{text}'")
    
    if show_positions:
        # Show character positions
        print("Positions:")
        print("0123456789" * (len(text) // 10 + 1))
        print(text)
        print()
    
    # Show each token with its boundaries
    print("Token boundaries:")
    for i, token in enumerate(sentence.tokens):
        # Create a visual representation
        visual = [' '] * len(text)
        for pos in range(token.start, token.end):
            visual[pos] = '█'
        
        print(f"Token[{i:2d}]: '{token.text}' [{token.start}:{token.end}]")
        print(f"          {''.join(visual)}")
    
    return text

def visualize_mwe_spans(sentence, mwe):
    """Visualize MWE token spans."""
    text = sentence.text
    
    print(f"\nMWE: '{mwe.lemma}' (type: {mwe.type})")
    print(f"Token indices: {mwe.token_indices}")
    
    # Show which characters are covered by MWE tokens
    mwe_coverage = [' '] * len(text)
    
    for token_idx in mwe.token_indices:
        token = sentence.tokens[token_idx]
        for pos in range(token.start, token.end):
            mwe_coverage[pos] = '█'
    
    print("MWE coverage:")
    print("0123456789" * (len(text) // 10 + 1))
    print(text)
    print(''.join(mwe_coverage))
    
    # Show span from min to max token position (potential error source)
    if mwe.token_indices:
        min_idx = min(mwe.token_indices)
        max_idx = max(mwe.token_indices)
        
        span_coverage = [' '] * len(text)
        for idx in range(min_idx, max_idx + 1):
            token = sentence.tokens[idx]
            for pos in range(token.start, token.end):
                if idx in mwe.token_indices:
                    span_coverage[pos] = '█'  # Correct MWE token
                else:
                    span_coverage[pos] = '?'  # Potentially wrong token
        
        print("\nFull span (min to max index) - '?' shows potential wrong tokens:")
        print(''.join(span_coverage))

# Visualize all sentences
if sentences:
    for sent_idx, sentence in enumerate(sentences):
        print(f"\n{'='*80}")
        print(f"SENTENCE {sent_idx + 1} VISUALIZATION")
        print(f"{'='*80}")
        
        visualize_sentence_tokens(sentence)
        
        for mwe_idx, mwe in enumerate(sentence.mwes):
            print(f"\n{'-'*40}")
            print(f"MWE {mwe_idx + 1} VISUALIZATION")
            print(f"{'-'*40}")
            visualize_mwe_spans(sentence, mwe)


SENTENCE 1 VISUALIZATION
Sentence: 'Bakterije se mogu širiti i putem krvi.'
Positions:
0123456789012345678901234567890123456789
Bakterije se mogu širiti i putem krvi.

Token boundaries:
Token[ 0]: 'Bakterije' [0:9]
          █████████                             
Token[ 1]: 'se' [10:12]
                    ██                          
Token[ 2]: 'mogu' [13:17]
                       ████                     
Token[ 3]: 'širiti' [18:24]
                            ██████              
Token[ 4]: 'i' [25:26]
                                   █            
Token[ 5]: 'putem' [27:32]
                                     █████      
Token[ 6]: 'krvi' [33:37]
                                           ████ 
Token[ 7]: '.' [37:38]
                                               █

----------------------------------------
MWE 1 VISUALIZATION
----------------------------------------

MWE: 'širiti se' (type: IRV)
Token indices: [1, 3]
MWE coverage:
0123456789012345678901234567890123456789
Bakte

## 6. Test MWE Marking Functions

Let's test the `mark_mwe()` and `mark_tokens()` functions to see how they handle MWE highlighting.

In [6]:
# Test marking functions
if sentences:
    for sent_idx, sentence in enumerate(sentences):
        print(f"\n{'='*60}")
        print(f"SENTENCE {sent_idx + 1} MARKING TEST")
        print(f"{'='*60}")
        print(f"Original: {sentence.text}")
        
        if not sentence.mwes:
            print("No MWEs to mark")
            continue
            
        for mwe_idx, mwe in enumerate(sentence.mwes):
            print(f"\n--- MWE {mwe_idx + 1}: '{mwe.lemma}' ---")
            
            try:
                # Test mark_mwe function
                marked_sentence = mark_mwe(mwe, sentence, "<<", ">>")
                print(f"Marked (mark_mwe): {marked_sentence}")
                
                # Test mark_tokens function with extracted tokens
                extracted_tokens = get_mwe_tokens(sentence, mwe)
                marked_tokens = mark_tokens(extracted_tokens, sentence, "[[", "]]")
                print(f"Marked (mark_tokens): {marked_tokens}")
                
                # Check if they produce the same result
                same_result = marked_sentence.replace("<<", "[[").replace(">>", "]]") == marked_tokens
                print(f"Same result: {same_result}")
                
                if not same_result:
                    print("⚠️  WARNING: Different results from mark_mwe vs mark_tokens!")
                    print("This might indicate the token extraction issue!")
                
                # Manual marking test - what would happen if we took all tokens from min to max index
                if mwe.token_indices:
                    min_idx = min(mwe.token_indices)
                    max_idx = max(mwe.token_indices)
                    wrong_tokens = [sentence.tokens[i] for i in range(min_idx, max_idx + 1)]
                    wrong_marked = mark_tokens(wrong_tokens, sentence, "{{", "}}")
                    print(f"Wrong (all span): {wrong_marked}")
                    
                    if wrong_marked != marked_tokens:
                        print("✅ Good: Function doesn't take all tokens in span")
                    else:
                        print("❌ BAD: Function takes all tokens in span!")
                
            except Exception as e:
                print(f"❌ Error in marking: {e}")
                import traceback
                traceback.print_exc()


SENTENCE 1 MARKING TEST
Original: Bakterije se mogu širiti i putem krvi.

--- MWE 1: 'širiti se' ---
Marked (mark_mwe): Bakterije <<se>> mogu <<širiti>> i putem krvi.
Marked (mark_tokens): Bakterije [[se]] mogu [[širiti]] i putem krvi.
Same result: True
Wrong (all span): Bakterije {{se}} {{mogu}} {{širiti}} i putem krvi.
✅ Good: Function doesn't take all tokens in span

SENTENCE 2 MARKING TEST
Original: Simptomi se obično poboljšavaju u roku od dva dana, ali mogu trajati i do sedam dana.

--- MWE 1: 'poboljšavati se' ---
Marked (mark_mwe): Simptomi <<se>> obično <<poboljšavaju>> u roku od dva dana, ali mogu trajati i do sedam dana.
Marked (mark_tokens): Simptomi [[se]] obično [[poboljšavaju]] u roku od dva dana, ali mogu trajati i do sedam dana.
Same result: True
Wrong (all span): Simptomi {{se}} {{obično}} {{poboljšavaju}} u roku od dva dana, ali mogu trajati i do sedam dana.
✅ Good: Function doesn't take all tokens in span

--- MWE 2: 'u roku od' ---
Marked (mark_mwe): Simptomi se o

## 7. Compare Expected vs Actual Results

Let's analyze the MWE structure and identify any discrepancies in token selection.

In [7]:
# Detailed analysis of MWE structure and potential issues
def analyze_mwe_structure(sentence, mwe):
    """Analyze MWE structure and identify potential issues."""
    
    print(f"MWE Analysis: '{mwe.lemma}' (type: {mwe.type})")
    print(f"Token indices: {mwe.token_indices}")
    
    # Check for gaps in token indices
    if len(mwe.token_indices) > 1:
        sorted_indices = sorted(mwe.token_indices)
        min_idx, max_idx = sorted_indices[0], sorted_indices[-1]
        
        print(f"Index range: {min_idx} to {max_idx}")
        
        # Check for gaps
        expected_continuous = list(range(min_idx, max_idx + 1))
        missing_indices = set(expected_continuous) - set(mwe.token_indices)
        extra_indices = set(mwe.token_indices) - set(expected_continuous)
        
        if missing_indices:
            print(f"❌ Missing indices (gaps in MWE): {sorted(missing_indices)}")
            print("Tokens in gaps:")
            for idx in sorted(missing_indices):
                token = sentence.tokens[idx]
                print(f"  Token[{idx}]: '{token.text}' [{token.start}:{token.end}]")
        
        if extra_indices:
            print(f"⚠️  Extra indices (non-continuous): {sorted(extra_indices)}")
        
        if not missing_indices and not extra_indices:
            print("✅ MWE tokens are continuous")
    
    # Show actual tokens
    print(f"\nActual MWE tokens:")
    for i, token_idx in enumerate(sorted(mwe.token_indices)):
        token = sentence.tokens[token_idx]
        print(f"  {i}: Token[{token_idx}] = '{token.text}' [{token.start}:{token.end}]")
    
    # Reconstruct MWE text
    mwe_tokens = [sentence.tokens[idx] for idx in sorted(mwe.token_indices)]
    
    # Method 1: Concatenate token texts
    reconstructed_text = ' '.join(token.text for token in mwe_tokens)
    
    # Method 2: Extract from original text using span
    if mwe_tokens:
        span_start = min(token.start for token in mwe_tokens)
        span_end = max(token.end for token in mwe_tokens)
        span_text = sentence.text[span_start:span_end]
        
        print(f"\nReconstruction comparison:")
        print(f"  Method 1 (join tokens): '{reconstructed_text}'")
        print(f"  Method 2 (text span): '{span_text}'")
        print(f"  MWE lemma: '{mwe.lemma}'")
        
        return {
            'mwe': mwe,
            'token_indices': mwe.token_indices,
            'has_gaps': len(missing_indices) > 0 if len(mwe.token_indices) > 1 else False,
            'reconstructed_text': reconstructed_text,
            'span_text': span_text,
            'lemma': mwe.lemma
        }

# Run analysis on all MWEs
analysis_results = []

if sentences:
    for sent_idx, sentence in enumerate(sentences):
        print(f"\n{'='*70}")
        print(f"DETAILED ANALYSIS - SENTENCE {sent_idx + 1}")
        print(f"{'='*70}")
        print(f"Text: {sentence.text}")
        
        if not sentence.mwes:
            print("No MWEs to analyze")
            continue
        
        for mwe_idx, mwe in enumerate(sentence.mwes):
            print(f"\n{'-'*50}")
            print(f"MWE {mwe_idx + 1} Analysis")
            print(f"{'-'*50}")
            
            result = analyze_mwe_structure(sentence, mwe)
            if result:
                analysis_results.append(result)

# Summary of findings
print(f"\n{'='*70}")
print(f"SUMMARY OF FINDINGS")
print(f"{'='*70}")

total_mwes = len(analysis_results)
mwes_with_gaps = sum(1 for r in analysis_results if r['has_gaps'])

print(f"Total MWEs analyzed: {total_mwes}")
print(f"MWEs with gaps: {mwes_with_gaps}")
print(f"MWEs without gaps: {total_mwes - mwes_with_gaps}")

if mwes_with_gaps > 0:
    print(f"\n❌ Found {mwes_with_gaps} MWEs with gaps - this could be the source of the problem!")
    print("The issue might be in how token indices are determined during parsing.")
else:
    print("\n✅ No gaps found in MWE token indices.")
    print("The issue might be elsewhere - possibly in the mark_tokens function or text reconstruction.")


DETAILED ANALYSIS - SENTENCE 1
Text: Bakterije se mogu širiti i putem krvi.

--------------------------------------------------
MWE 1 Analysis
--------------------------------------------------
MWE Analysis: 'širiti se' (type: IRV)
Token indices: [1, 3]
Index range: 1 to 3
❌ Missing indices (gaps in MWE): [2]
Tokens in gaps:
  Token[2]: 'mogu' [13:17]

Actual MWE tokens:
  0: Token[1] = 'se' [10:12]
  1: Token[3] = 'širiti' [18:24]

Reconstruction comparison:
  Method 1 (join tokens): 'se širiti'
  Method 2 (text span): 'se mogu širiti'
  MWE lemma: 'širiti se'

DETAILED ANALYSIS - SENTENCE 2
Text: Simptomi se obično poboljšavaju u roku od dva dana, ali mogu trajati i do sedam dana.

--------------------------------------------------
MWE 1 Analysis
--------------------------------------------------
MWE Analysis: 'poboljšavati se' (type: IRV)
Token indices: [1, 3]
Index range: 1 to 3
❌ Missing indices (gaps in MWE): [2]
Tokens in gaps:
  Token[2]: 'obično' [12:18]

Actual MWE tokens:
 

## 8. Test Collect MWE Senses Function

Let's also test the `collect_mwe_senses` function to see how it behaves with the extracted tokens.

In [8]:
# Create a dummy sense repository for testing
dummy_sense_df = pd.DataFrame({
    'lemma': ['širiti se', 'poboljšavati se', 'u roku od', 'bakterija', 'simptom'],
    'type': ['IRV', 'IRV', 'AdpID', 'N', 'N'],
    'sense_id': ['CVMWE-0399', 'CVMWE-0292', 'LLM-0287', 'ENG30-01348530-n', 'ENG30-14299637-n']
})

print("Dummy sense repository:")
print(dummy_sense_df)
print()

# Test collect_mwe_senses function
if sentences:
    print("Testing collect_mwe_senses function:")
    print("="*50)
    
    try:
        mwe_results = collect_mwe_senses(sentences, dummy_sense_df)
        
        print(f"Found {len(mwe_results)} MWE results")
        
        for i, result in enumerate(mwe_results):
            print(f"\n--- MWE Result {i+1} ---")
            print(f"Lemma: '{result['lemma']}'")
            print(f"Type: '{result['type']}'")
            print(f"Number of senses: {result['num_senses']}")
            print(f"Marked sentence: {result['marked_sentence']}")
            print(f"Original sentence: {result['sentence']}")
            
            # Show the senses found
            if result['num_senses'] > 0:
                print("Matching senses:")
                print(result['senses'])
            
    except Exception as e:
        print(f"❌ Error testing collect_mwe_senses: {e}")
        import traceback
        traceback.print_exc()

Dummy sense repository:
             lemma   type          sense_id
0        širiti se    IRV        CVMWE-0399
1  poboljšavati se    IRV        CVMWE-0292
2        u roku od  AdpID          LLM-0287
3        bakterija      N  ENG30-01348530-n
4          simptom      N  ENG30-14299637-n

Testing collect_mwe_senses function:
Found 3 MWE results

--- MWE Result 1 ---
Lemma: 'širiti se'
Type: 'IRV'
Number of senses: 1
Marked sentence: Bakterije **se** mogu **širiti** i putem krvi.
Original sentence: Bakterije se mogu širiti i putem krvi.
Matching senses:
       lemma type    sense_id
0  širiti se  IRV  CVMWE-0399

--- MWE Result 2 ---
Lemma: 'poboljšavati se'
Type: 'IRV'
Number of senses: 1
Marked sentence: Simptomi **se** obično **poboljšavaju** u roku od dva dana, ali mogu trajati i do sedam dana.
Original sentence: Simptomi se obično poboljšavaju u roku od dva dana, ali mogu trajati i do sedam dana.
Matching senses:
             lemma type    sense_id
1  poboljšavati se  IRV  CVMWE-029

## 9. Conclusions and Next Steps

Based on our analysis, we can identify where the MWE token extraction issue occurs and propose solutions.

In [9]:
print("🔍 DEBUG ANALYSIS COMPLETE")
print("="*50)

print("\nKey findings from this debug session:")

print("\n1. FUNCTION TESTING:")
print("   - get_mwe_tokens(): Tests if this function correctly extracts only MWE tokens")
print("   - mark_mwe() and mark_tokens(): Tests if highlighting functions work correctly") 
print("   - collect_mwe_senses(): Tests the main processing pipeline")

print("\n2. DATA STRUCTURE ANALYSIS:")
print("   - Token indices in MWE objects")
print("   - Token boundaries and character positions")
print("   - Gaps or overlaps in token sequences")

print("\n3. POTENTIAL ISSUE SOURCES:")
print("   - MWE parsing during TSV loading (data_loader.py)")
print("   - Token index calculation in MultiWordExpression objects")
print("   - Text span calculation in mark_tokens() function")
print("   - Range-based token extraction (taking min to max index)")

print("\n4. NEXT STEPS:")
if analysis_results:
    gaps_found = any(r['has_gaps'] for r in analysis_results)
    if gaps_found:
        print("   ❌ GAPS DETECTED: Check TSV parsing logic in data_loader.py")
        print("   ❌ Verify MWE token index assignment during annotation loading")
    else:
        print("   ✅ No gaps in token indices - issue is likely in processing functions")
        print("   🔍 Check mark_tokens() function for span-based text extraction")
        print("   🔍 Verify get_mwe_tokens() implementation")

print("\n5. DEBUGGING RECOMMENDATIONS:")
print("   - Run this notebook with your actual data files")
print("   - Check WebAnno TSV format compliance") 
print("   - Examine data_loader.py parsing logic")
print("   - Test with different MWE types (continuous vs non-continuous)")
print("   - Validate token boundary calculations")

print(f"\n📊 Analysis completed for {len(analysis_results)} MWEs")
print("Use the visualizations above to identify specific problem cases.")

# Save analysis results for further investigation
if analysis_results:
    results_df = pd.DataFrame(analysis_results)
    print(f"\n💾 Analysis results saved as DataFrame:")
    print(results_df[['lemma', 'has_gaps', 'reconstructed_text']].to_string())

🔍 DEBUG ANALYSIS COMPLETE

Key findings from this debug session:

1. FUNCTION TESTING:
   - get_mwe_tokens(): Tests if this function correctly extracts only MWE tokens
   - mark_mwe() and mark_tokens(): Tests if highlighting functions work correctly
   - collect_mwe_senses(): Tests the main processing pipeline

2. DATA STRUCTURE ANALYSIS:
   - Token indices in MWE objects
   - Token boundaries and character positions
   - Gaps or overlaps in token sequences

3. POTENTIAL ISSUE SOURCES:
   - MWE parsing during TSV loading (data_loader.py)
   - Token index calculation in MultiWordExpression objects
   - Text span calculation in mark_tokens() function
   - Range-based token extraction (taking min to max index)

4. NEXT STEPS:
   ❌ GAPS DETECTED: Check TSV parsing logic in data_loader.py
   ❌ Verify MWE token index assignment during annotation loading

5. DEBUGGING RECOMMENDATIONS:
   - Run this notebook with your actual data files
   - Check WebAnno TSV format compliance
   - Examine data